# D4 federated DP-FedAvg — raw EEG never leaves the device

Central-DP FedAvg with 104 clients (one per PhysioNet subject). 30 rounds, 50% participation per round, 3 local SGD epochs, server clips per-client deltas to ‖Δ‖₂ ≤ 1, sums, adds Gaussian noise σ=0.4, divides by participant count. After training we run A1 closed-set + encoder-fine-tune attacks.

The deployment story this closes: a BCI service where raw EEG stays on the user device. We benchmark whether participant-level DP (each user appears once) provides comparable empirical privacy to per-sample DP-SGD.

**Runtime → L4 GPU.** Expected wall: ~60-90 min.

**Don't `Save a copy in GitHub` from Colab.**

In [ ]:
!rm -rf /content/bci-identity-leakage
!git clone https://github.com/manrajmondair/bci-identity-leakage.git /content/bci-identity-leakage
%cd /content/bci-identity-leakage
!git rev-parse HEAD
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
import torch
tv = torch.__version__.split('+')[0]
!pip install --quiet "torchaudio=={tv}"
!pip install --quiet mne moabb pyriemann braindecode opacus httpx tqdm rich scipy

In [ ]:
!python -m data.prefetch_direct --runs imagery --workers 8

In [ ]:
!PYTHONUNBUFFERED=1 python -u -m experiments.31_federated_dp --all

In [ ]:
import json, os, subprocess, datetime, platform, sys
import torch
PROJECT_DIR = '/content/bci-identity-leakage'
if os.path.isdir(PROJECT_DIR): os.chdir(PROJECT_DIR)
RESULTS = 'results/31_federated_dp.json'
TAG = 'fed_dp'
try:
    git_sha = subprocess.check_output(['git','rev-parse','HEAD']).decode().strip()
except Exception: git_sha = 'unknown'
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
now_utc = datetime.datetime.utcnow().replace(microsecond=0).isoformat() + 'Z'
run_id = now_utc.replace(':','').replace('-','').rstrip('Z') + f'_{TAG}_{git_sha[:7]}'
meta = {'run_id': run_id, 'experiment': 'experiments.31_federated_dp',
        'git_sha': git_sha, 'hardware': f'Colab {gpu}',
        'platform': platform.platform(), 'torch_version': torch.__version__,
        'completed_at_utc': now_utc, 'outputs': [RESULTS]}
os.makedirs(f'runs/{run_id}', exist_ok=True)
with open(f'runs/{run_id}/meta.json','w') as f: json.dump(meta, f, indent=2)
if not os.path.exists(RESULTS):
    sys.exit(f'!!! {RESULTS} missing.')
print('--- BEGIN 31_federated_dp.json ---')
print(open(RESULTS).read())
print('--- END 31_federated_dp.json ---')
print()
print('--- BEGIN run meta ---')
print(json.dumps(meta, indent=2))
print('--- END run meta ---')